In [ ]:
import requests

url = "http://node880:3000/v1/chat/completions"
payload = {
    "model": "qwen3-0.6b-base",
    "messages": [{"role": "user", "content": "Say ok in one word."}],
    "max_tokens": 32,
}
r = requests.post(url, json=payload, timeout=120)
r.raise_for_status()
print(r.json())

{'id': '8e3e983d0db6453eadb3948071e7a003', 'object': 'chat.completion', 'created': 1774637935, 'model': 'qwen3-0.6b-base', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Say ok in one word. للغا\n⋯⋯\n…⋯saying…\n…the…\n…the…\n…for…\n…of…\n…is…\n…is', 'reasoning_content': None, 'tool_calls': None}, 'logprobs': None, 'finish_reason': 'length', 'matched_stop': None}], 'usage': {'prompt_tokens': 14, 'total_tokens': 46, 'completion_tokens': 32, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'metadata': {'weight_version': 'default'}}


In [7]:
from openai import OpenAI

client = OpenAI(base_url="http://node880:3000/v1", api_key="not-needed")
resp = client.chat.completions.create(
    model="qwen3-0.6b-base",
    messages=[{"role": "user", "content": "Say ok in one word."}],
    max_tokens=32,
)
print(resp.choices[0].message.content)

Say ok in one word.ических
Say ok in one word.ication
Say ok in one word.ication
Say ok in one word.ication



In [2]:
import json
import os
import socket
import requests

# 127.0.0.1 = the machine running *this Python kernel*, not necessarily where you ran curl in SSH.
# If unset, we try localhost then node880. To force one URL: export OLLAMA_BASE=http://host:11434
KERNEL_HOST = socket.gethostname()


def _ollama_tags_ok(base: str) -> bool:
    try:
        r = requests.get(f"{base.rstrip('/')}/api/tags", timeout=4)
        return r.ok
    except requests.RequestException:
        return False


_explicit = os.environ.get("OLLAMA_BASE")
if _explicit:
    OLLAMA_BASE = _explicit.rstrip("/")
    if not _ollama_tags_ok(OLLAMA_BASE):
        raise RuntimeError(
            f"OLLAMA_BASE={OLLAMA_BASE!r} not reachable from kernel host {KERNEL_HOST!r}. "
            "Check URL, firewall, and that ollama serve is running."
        )
else:
    OLLAMA_BASE = None
    for candidate in ("http://127.0.0.1:11434", "http://node880:11434"):
        if _ollama_tags_ok(candidate):
            OLLAMA_BASE = candidate
            break
    if OLLAMA_BASE is None:
        raise RuntimeError(
            f"No Ollama on 127.0.0.1:11434 or node880:11434 from kernel host {KERNEL_HOST!r}. "
            "Your working curl was on another machine.\n"
            "Pick one: (1) ssh -L 11434:127.0.0.1:11434 you@node880  then export OLLAMA_BASE=http://127.0.0.1:11434\n"
            "  (2) On the Ollama host: OLLAMA_HOST=0.0.0.0:11434 ollama serve  then export OLLAMA_BASE=http://node880:11434\n"
            "  (3) Run Jupyter on the same node as ollama serve."
        )

print(f"OLLAMA_BASE={OLLAMA_BASE}  (kernel: {KERNEL_HOST})")
MODEL = os.environ.get("OLLAMA_MODEL", "qwen3.5:9b")

# --- Chat API (messages, like OpenAI chat) ---
chat_url = f"{OLLAMA_BASE}/api/chat"
chat_payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "Say ok in one word."},
    ],
    "stream": False,
    "options": {"temperature": 0},
}
r = requests.post(chat_url, json=chat_payload, timeout=120)
r.raise_for_status()
data = r.json()
print(json.dumps(data, indent=2)[:2000])
print("--- reply ---")
print(data.get("message", {}).get("content", ""))

# --- Generate API (single prompt string) ---
gen_url = f"{OLLAMA_BASE}/api/generate"
gen_payload = {
    "model": MODEL,
    "prompt": "Say ok in one word.",
    "stream": False,
}
r2 = requests.post(gen_url, json=gen_payload, timeout=120)
r2.raise_for_status()
gen_data = r2.json()
print("--- generate response ---")
print(gen_data.get("response", ""))

OLLAMA_BASE=http://node880:11434  (kernel: nodelogin03.alice.leidenuniv.nl)
{
  "model": "qwen3.5:9b",
  "created_at": "2026-03-28T00:00:08.79515171Z",
  "message": {
    "role": "assistant",
    "content": "ok",
    "thinking": "Thinking Process:\n\n1.  **Analyze the Request:** The user is asking me to say \"ok\" in one word.\n2.  **Identify the Constraint:** The response must be exactly one word.\n3.  **Determine the Content:** The content should be \"ok\" (or \"Okay\", but \"ok\" is more standard for \"one word\" requests).\n4.  **Check for Safety/Policy:** There are no safety concerns with saying \"ok\".\n5.  **Formulate Output:** The output should be just \"ok\".\n6.  **Final Review:** Does \"ok\" count as one word? Yes. Does it satisfy the prompt? Yes.\n\nDecision: Output \"ok\"."
  },
  "done": true,
  "done_reason": "stop",
  "total_duration": 6738165589,
  "load_duration": 2489722432,
  "prompt_eval_count": 16,
  "prompt_eval_duration": 220372070,
  "eval_count": 158,
  "eval_